# 03 Generation runner: smoke test (Prompt 09, cycle 3)

**Needs a GPU**: Runtime → Change runtime type → **T4 GPU**. First upload into the Drive project folder: `src/generate.py`, `scripts/run_generation.py`, `tests/test_generate.py`, `notebooks/03_generation_smoke_test.ipynb`. Then Runtime → **Run all** (about 10–12 minutes, most of it the model download). Uses **development items only**; no test-pool item is touched. Copy all the output back into the chat.

In [ ]:
# 1. Drive, project folder, libraries
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/atr_colab/AITR_Lab02_MCL2026002_PromptStability
!pip install -q "transformers>=4.37.0" accelerate bitsandbytes pyyaml
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Unit tests (fake model, no GPU needed for these)
!python tests/test_generate.py 2>&1 | grep -E 'PASS|FAIL|SUMMARY'

In [ ]:
# 3. The guard must refuse a test-pool condition before the freeze (an error here is the expected result)
!python scripts/run_generation.py --condition b0 --limit 1 2>&1 | tail -1

In [ ]:
# 4. Real model, development items only: 3 greedy + 4 sampled P0 generations
!python scripts/run_generation.py --condition pilot_greedy --prompts P0 --limit 3
!python scripts/run_generation.py --condition pilot_sampled --prompts P0 --limit 4

In [ ]:
# 5. Resume check: the same command again must skip all 3 and generate 0
!python scripts/run_generation.py --condition pilot_greedy --prompts P0 --limit 3

In [ ]:
# 6. GSM8K path: 2 greedy chain-of-thought generations
!python scripts/run_generation.py --condition gsm8k_greedy --limit 2

In [ ]:
# 7. What was saved (raw text shortened)
import json, glob
for path in sorted(glob.glob('results/generations/*.jsonl')):
    print('=====', path)
    for line in open(path):
        r = json.loads(line)
        print(r['run_id'], '| seed', r['seed'], '|', r['parse_status'], '|', r.get('decision', r.get('answer')),
              '| out tokens', r['n_out_tokens'], '|', r['latency_s'], 's')
        print('   ', r['raw_text'][:220].replace('\n', ' '))